[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_LLM_Workshop/NN_DL_LLM_NanoGPT/NN_DL_LLM_NanoGPT.ipynb)

# Building a Large Language Model from Scratch

**NanoGPT: A minimal GPT built from scratch in ~300 lines of PyTorch**

**Author:** Daniel Traian Pele, ASE Bucharest / IDA

**Keywords:** GPT, transformer, attention, language model, nanoGPT, from scratch


In [ ]:
!pip install torch matplotlib numpy -q


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


## 1. Download Shakespeare Corpus


In [ ]:
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'input.txt')
with open('input.txt', 'r') as f:
    text = f.read()
print(f'Corpus: {len(text):,} characters')
print(f'First 200 chars:\n{text[:200]}')


## 2. Character-Level Tokenizer


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode('hello'), '->', decode(encode('hello')))


## 3. Data Pipeline


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, batch_size=32, block_size=128):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

print(f'Train: {len(train_data):,} tokens, Val: {len(val_data):,} tokens')


## 4. Token + Positional Embeddings


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        B, T = x.shape
        return self.tok_emb(x) + self.pos_emb(torch.arange(T, device=x.device))


## 5. Self-Attention (Single Head)


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.W_q = nn.Linear(d_model, head_dim, bias=False)
        self.W_k = nn.Linear(d_model, head_dim, bias=False)
        self.W_v = nn.Linear(d_model, head_dim, bias=False)
        self.head_dim = head_dim
    def forward(self, x, mask=None):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = Q @ K.transpose(-2, -1) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        return weights @ V, weights


## 6. Multi-Head Attention


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.head_dim = d_model // n_heads
        self.heads = nn.ModuleList([SelfAttention(d_model, self.head_dim) for _ in range(n_heads)])
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x, mask=None):
        out = torch.cat([h(x, mask)[0] for h in self.heads], dim=-1)
        return self.proj(out)


## 7. Transformer Block


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x


## 8. Complete NanoGPT Model


In [ ]:
class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.embedding = TokenEmbedding(vocab_size, d_model, max_seq_len)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.register_buffer('mask', torch.tril(torch.ones(max_seq_len, max_seq_len)))
    def forward(self, x, targets=None):
        B, T = x.shape
        h = self.embedding(x)
        mask = self.mask[:T, :T]
        for block in self.blocks:
            h = block(h, mask)
        logits = self.head(self.ln_f(h))
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss
    @torch.no_grad()
    def generate(self, x, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            x_cond = x[:, -self.max_seq_len:]
            logits, _ = self(x_cond)
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            x = torch.cat([x, torch.multinomial(probs, 1)], dim=1)
        return x

model = NanoGPT(vocab_size, d_model=64, n_heads=4, n_layers=4, d_ff=256, max_seq_len=128).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print('Before training:', decode(model.generate(torch.zeros((1,1), dtype=torch.long, device=device), 100)[0].tolist()))


## 9. Training Loop


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
train_losses, val_losses = [], []
for step in range(5000):
    model.train()
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 500 == 0:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch('val')
            _, vl = model(xv, yv)
        train_losses.append(loss.item()); val_losses.append(vl.item())
        print(f'Step {step:5d} | train {loss.item():.3f} | val {vl.item():.3f}')


## 10. Training Curve


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
steps = np.arange(len(train_losses)) * 500
ax.plot(steps, train_losses, color='#1A3A6E', lw=2, label='Train')
ax.plot(steps, val_losses, color='#CD0000', lw=2, label='Val')
ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title('NanoGPT Training', fontweight='bold')
ax.set_facecolor('none'); fig.patch.set_alpha(0)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), fontsize=8, ncol=2, frameon=False)
plt.savefig('nn_llm_training_curve.pdf', bbox_inches='tight', dpi=300, transparent=True)
plt.savefig('nn_llm_training_curve.png', bbox_inches='tight', dpi=150, transparent=True)
plt.show()


## 11. Generate Text at Different Temperatures


In [ ]:
model.eval()
prompt = torch.zeros((1, 1), dtype=torch.long, device=device)
for temp in [0.5, 0.8, 1.0, 1.5]:
    print(f'\n{"="*60}\nTemperature = {temp}\n{"="*60}')
    out = model.generate(prompt, max_new_tokens=200, temperature=temp)
    print(decode(out[0].tolist()))
